# MCI-GRU Recommended Confirmation Ablation - Google Colab

Focused follow-up to the dynamic graph and training-factor ablation notebooks. This notebook tests the report recommendations without re-running a broad search.

## 1. Mount Drive, Clone Repo, Install Dependencies

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount('/content/drive')

REPO_URL = 'https://github.com/magilliam27/MCI-GRU.git'
BRANCH = 'main'
REPO_DIR = Path('/content/MCI-GRU')
DRIVE_ROOT = Path('/content/drive/MyDrive/MCI-GRU-Ablations')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    os.chdir(REPO_DIR)
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e ".[dev,tracking,fred]"

REQUIRE_GPU = True

try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is not installed; rerun this setup cell.') from exc

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(device_idx))
    print(f'GPU memory: {props.total_memory / (1024 ** 3):.1f} GiB')
else:
    msg = (
        'No CUDA GPU is visible in the notebook kernel. In Colab, choose '
        'Runtime -> Change runtime type -> GPU, then restart and run from the top.'
    )
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print('WARNING:', msg)

child_probe = subprocess.run(
    [
        sys.executable,
        '-c',
        "import torch; "
        "print('Child Torch:', torch.__version__); "
        "print('Child CUDA available:', torch.cuda.is_available()); "
        "print('Child CUDA version:', torch.version.cuda); "
        "print('Child GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO_GPU')",
    ],
    text=True,
    capture_output=True,
)
print('Child Python:', sys.executable)
print(child_probe.stdout.strip())
if child_probe.returncode != 0:
    print(child_probe.stderr)
    raise RuntimeError('Child Python CUDA probe failed; rerun setup after fixing PyTorch.')
child_cuda_visible = 'Child CUDA available: True' in child_probe.stdout
if REQUIRE_GPU and not child_cuda_visible:
    msg = (
        'The training subprocess cannot see CUDA, so run_experiment.py would train on CPU. '
        'Restart the Colab GPU runtime and rerun setup; if this persists, reinstall a CUDA-enabled torch wheel.'
    )
    raise RuntimeError(msg)

print(f'Repo: {REPO_DIR}')
print(f'Drive outputs: {DRIVE_ROOT}')

## 2. Confirmation Run Configuration

In [ ]:
from datetime import datetime

DATA_FILE_CANDIDATES = [
    'sp500_2019_universe_data_through_2026.csv',
    'sp500_2019_universe_data.csv',
    'sp500_data.csv',
]
GDRIVE_DATA_DIR = Path('/content/drive/MyDrive/MCI-GRU-Data')

TRAIN_START = '2019-01-01'
TRAIN_END = '2023-12-31'
VAL_START = '2024-01-08'
VAL_END = '2024-12-31'
TEST_START = '2025-01-08'
TEST_END = '2025-12-31'

NUM_MODELS = 20
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
BOOTSTRAP_RESAMPLES = 1000

BATCH_SIZE = 32
LEARNING_RATE = '5e-5'
SEEDS = [42]  # For confirmation replication, use [42, 43, 44].

REGIME_STRICT_FOR_REGIME_RUNS = True
REGIME_INPUTS_CSV = ''  # Optional path copied into the repo, e.g. data/raw/market/regime_inputs.csv.
REGIME_ENFORCE_LAG_DAYS = 0
FRED_API_KEY = ''
os.environ['FRED_API_KEY'] = FRED_API_KEY

INCLUDE_EDGE_DROPOUT_PROBES = False
EDGE_DROPOUT_PROBE_VALUES = [0.05, 0.2]
SKIP_COMPLETED_RUNS = True
RUN_ONLY_FAMILIES = []  # Example: ['primary'] or ['edge_dropout_probe']; empty means all queued runs.
RUN_NAME_CONTAINS = ''
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')

ABLATION_ROOT = DRIVE_ROOT / 'recommended_confirmation_ablation' / RUN_TAG
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)

print('Ablation root:', ABLATION_ROOT)
print('Models:', NUM_MODELS)
print('Epochs:', NUM_EPOCHS)
print('Early stopping patience:', EARLY_STOPPING_PATIENCE)
print('Bootstrap resamples:', BOOTSTRAP_RESAMPLES)
print('Seeds:', SEEDS)

## 3. Data Availability Check

In [ ]:
import pandas as pd
import shutil

DATA_FILE = None
for candidate in DATA_FILE_CANDIDATES:
    repo_data = REPO_DIR / candidate
    drive_data = GDRIVE_DATA_DIR / candidate
    if not repo_data.exists() and drive_data.exists():
        shutil.copy2(drive_data, repo_data)
    if repo_data.exists():
        DATA_FILE = candidate
        break

if DATA_FILE is None:
    searched = [str(REPO_DIR / c) for c in DATA_FILE_CANDIDATES] + [str(GDRIVE_DATA_DIR / c) for c in DATA_FILE_CANDIDATES]
    raise FileNotFoundError('Missing market data. Looked for: ' + ', '.join(searched))

repo_data = REPO_DIR / DATA_FILE
preview = pd.read_csv(repo_data, usecols=['dt', 'kdcode'])
preview['dt'] = pd.to_datetime(preview['dt'])
print('Using data file:', repo_data)
print(f'Rows: {len(preview):,}')
print(f'Stocks: {preview.kdcode.nunique():,}')
print(f'Dates: {preview.dt.min().date()} to {preview.dt.max().date()}')
del preview

if REGIME_INPUTS_CSV:
    regime_src = Path(REGIME_INPUTS_CSV)
    regime_repo_path = REPO_DIR / REGIME_INPUTS_CSV
    regime_drive_path = GDRIVE_DATA_DIR / regime_src.name
    if not regime_repo_path.exists() and regime_drive_path.exists():
        regime_repo_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(regime_drive_path, regime_repo_path)
    if not regime_repo_path.exists():
        raise FileNotFoundError(f'REGIME_INPUTS_CSV was set but not found: {regime_repo_path}')
    regime_preview = pd.read_csv(regime_repo_path, nrows=5)
    print('Using regime inputs CSV:', regime_repo_path)
    print('Regime columns:', list(regime_preview.columns))
elif REGIME_STRICT_FOR_REGIME_RUNS and not os.environ.get('FRED_API_KEY'):
    print('WARNING: Regime runs are strict, but FRED_API_KEY is empty and REGIME_INPUTS_CSV is unset.')
    print('Set one of those before running the matrix if you want regime runs to complete.')

## 4. Define Recommended Confirmation Matrix

In [ ]:
import itertools
import json

STATIC_MOMENTUM_OVERRIDES = [
    'features.include_momentum=true',
    'features.include_weekly_momentum=true',
    'features.momentum_encoding=binary',
    'features.momentum_blend_mode=static',
    'features.momentum_blend_fast_weight=0.5',
]

COMMON_GRAPH_OVERRIDES = [
    'graph.judge_value=0.8',
    'graph.top_k=0',
    'graph.top_k_metric=corr',
    'graph.use_multi_feature_edges=true',
    'graph.append_snapshot_age_days=false',
    'graph.use_lead_lag_features=false',
    'training.shuffle_train=true',
]

GRAPH_FACTORS = {
    'static_threshold_shuffle': {
        'description': 'Static threshold graph; robust baseline from the dynamic-graph notebook.',
        'update_frequency_months': 0,
        'corr_lookback_days': 252,
        'overrides': [*COMMON_GRAPH_OVERRIDES, 'graph.update_frequency_months=0', 'graph.corr_lookback_days=252'],
    },
    'dynamic_threshold_6mo_lookback504_shuffle': {
        'description': 'Dynamic threshold graph every 6 months with a 504-day correlation lookback; best dynamic challenger.',
        'update_frequency_months': 6,
        'corr_lookback_days': 504,
        'overrides': [*COMMON_GRAPH_OVERRIDES, 'graph.update_frequency_months=6', 'graph.corr_lookback_days=504'],
    },
}

OBJECTIVE_FACTORS = {
    'pure_ic_returns_5d_val_ic': {
        'description': 'Pure IC loss on raw 5-day return labels, selected by validation IC.',
        'loss_type': 'ic',
        'ic_loss_alpha': None,
        'label_type': 'returns',
        'label_t': 5,
        'selection_metric': 'val_ic',
        'overrides': ['training.loss_type=ic', 'training.label_type=returns', 'model.label_t=5', 'training.selection_metric=val_ic'],
    },
    'combined_alpha075_returns_5d_val_loss': {
        'description': 'Combined MSE + IC loss with alpha 0.75, selected by validation loss.',
        'loss_type': 'combined',
        'ic_loss_alpha': 0.75,
        'label_type': 'returns',
        'label_t': 5,
        'selection_metric': 'val_loss',
        'overrides': ['training.loss_type=combined', 'training.ic_loss_alpha=0.75', 'training.label_type=returns', 'model.label_t=5', 'training.selection_metric=val_loss'],
    },
}

def regime_overrides(enabled: bool, include_subsequent_returns: bool) -> list[str]:
    overrides = [
        f'features.include_global_regime={str(enabled).lower()}',
        f'features.regime_strict={str(REGIME_STRICT_FOR_REGIME_RUNS and enabled).lower()}',
    ]
    if enabled:
        overrides.extend([
            f'features.regime_include_subsequent_returns={str(include_subsequent_returns).lower()}',
            'features.regime_subsequent_return_horizons=[1,3]',
            'features.regime_change_months=12',
            'features.regime_norm_months=120',
            'features.regime_exclusion_months=1',
            'features.regime_similarity_quantile=0.2',
            'features.regime_min_history_months=24',
            f'features.regime_enforce_lag_days={REGIME_ENFORCE_LAG_DAYS}',
        ])
        if REGIME_INPUTS_CSV:
            overrides.append(f'features.regime_inputs_csv={REGIME_INPUTS_CSV}')
    return overrides

REGIME_FACTORS = {
    'regime_current_only': {
        'description': 'Global regime similarity variables without similar-history forward-return context.',
        'overrides': regime_overrides(True, False),
    },
    'regime_with_forward_context': {
        'description': 'Global regime similarity variables plus similar-history subsequent-return context.',
        'overrides': regime_overrides(True, True),
    },
}

def clean_name(*parts):
    return '__'.join(part.replace('_', '-') for part in parts)

PRIMARY_CANDIDATES = [
    ('static_threshold_shuffle', 'pure_ic_returns_5d_val_ic', 'regime_current_only'),
    ('static_threshold_shuffle', 'pure_ic_returns_5d_val_ic', 'regime_with_forward_context'),
    ('dynamic_threshold_6mo_lookback504_shuffle', 'pure_ic_returns_5d_val_ic', 'regime_current_only'),
    ('dynamic_threshold_6mo_lookback504_shuffle', 'pure_ic_returns_5d_val_ic', 'regime_with_forward_context'),
    ('static_threshold_shuffle', 'combined_alpha075_returns_5d_val_loss', 'regime_with_forward_context'),
    ('dynamic_threshold_6mo_lookback504_shuffle', 'combined_alpha075_returns_5d_val_loss', 'regime_with_forward_context'),
]

def make_ablation(graph_key, objective_key, regime_key, seed, *, family='primary', drop_edge_p=0.1):
    graph = GRAPH_FACTORS[graph_key]
    objective = OBJECTIVE_FACTORS[objective_key]
    regime = REGIME_FACTORS[regime_key]
    return {
        'name': clean_name(graph_key, objective_key, regime_key, f'seed_{seed}', f'drop_edge_{str(drop_edge_p).replace(".", "p")}'),
        'description': ' | '.join([graph['description'], objective['description'], regime['description'], f'edge dropout p={drop_edge_p}', f'seed={seed}']),
        'family': family,
        'graph_factor': graph_key,
        'objective_factor': objective_key,
        'regime_factor': regime_key,
        'update_frequency_months': graph['update_frequency_months'],
        'corr_lookback_days': graph['corr_lookback_days'],
        'top_k': 0,
        'top_k_metric': 'corr',
        'edge_feature_set': 'corr_multi4',
        'loss_type': objective['loss_type'],
        'ic_loss_alpha': objective['ic_loss_alpha'],
        'label_type': objective['label_type'],
        'label_t': objective['label_t'],
        'selection_metric': objective['selection_metric'],
        'drop_edge_p': drop_edge_p,
        'seed': seed,
        'overrides': [
            *STATIC_MOMENTUM_OVERRIDES,
            *graph['overrides'],
            f'graph.drop_edge_p={drop_edge_p}',
            *objective['overrides'],
            *regime['overrides'],
            f'seed={seed}',
        ],
    }

ABLATIONS = []
for seed in SEEDS:
    for graph_key, objective_key, regime_key in PRIMARY_CANDIDATES:
        ABLATIONS.append(make_ablation(graph_key, objective_key, regime_key, seed))

if INCLUDE_EDGE_DROPOUT_PROBES:
    for seed, drop_edge_p in itertools.product(SEEDS, EDGE_DROPOUT_PROBE_VALUES):
        ABLATIONS.append(make_ablation(
            'static_threshold_shuffle',
            'pure_ic_returns_5d_val_ic',
            'regime_current_only',
            seed,
            family='edge_dropout_probe',
            drop_edge_p=drop_edge_p,
        ))

print('Primary candidates:', sum(a['family'] == 'primary' for a in ABLATIONS))
print('Edge-dropout probes:', sum(a['family'] == 'edge_dropout_probe' for a in ABLATIONS))
print('Total queued runs before filters:', len(ABLATIONS))
pd.DataFrame(ABLATIONS)[[
    'name', 'family', 'graph_factor', 'objective_factor', 'regime_factor',
    'loss_type', 'ic_loss_alpha', 'label_type', 'label_t', 'selection_metric',
    'drop_edge_p', 'seed', 'description'
]]

## 5. Helpers: Run, Collect, Score

In [ ]:
import numpy as np
from pathlib import Path

BASE_OVERRIDES = [
    'features=with_momentum',
    'data.source=csv',
    f'data.filename={DATA_FILE}',
    f'data.train_start={TRAIN_START}',
    f'data.train_end={TRAIN_END}',
    f'data.val_start={VAL_START}',
    f'data.val_end={VAL_END}',
    f'data.test_start={TEST_START}',
    f'data.test_end={TEST_END}',
    f'training.num_models={NUM_MODELS}',
    f'training.num_epochs={NUM_EPOCHS}',
    f'training.early_stopping_patience={EARLY_STOPPING_PATIENCE}',
    f'training.batch_size={BATCH_SIZE}',
    f'training.learning_rate={LEARNING_RATE}',
    f'evaluation.bootstrap_resamples={BOOTSTRAP_RESAMPLES}',
    'tracking.enabled=true',
    'tracking.log_predictions=false',
]

def run_ablation(ablation):
    run_dir = ABLATION_ROOT / ablation['name']
    run_dir.mkdir(parents=True, exist_ok=True)
    done_marker = run_dir / 'evaluation_summary.json'
    if SKIP_COMPLETED_RUNS and done_marker.exists():
        print('\n' + '=' * 100)
        print('Skipping completed run:', ablation['name'])
        print('Run dir:', run_dir)
        print('=' * 100)
        return {'name': ablation['name'], 'run_dir': str(run_dir), 'returncode': 0, 'skipped': True}

    cmd = [
        sys.executable,
        '-u',
        'run_experiment.py',
        *BASE_OVERRIDES,
        *ablation['overrides'],
        f'experiment_name={ablation["name"]}',
        f'hydra.run.dir={run_dir}',
    ]
    print('\n' + '=' * 100)
    print('Running:', ablation['name'])
    print(ablation['description'])
    print('Command:', ' '.join(cmd))
    print('=' * 100)
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    (run_dir / 'stdout.log').write_text(result.stdout)
    (run_dir / 'stderr.log').write_text(result.stderr)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return {'name': ablation['name'], 'run_dir': str(run_dir), 'returncode': result.returncode, 'skipped': False}

def flatten_json(prefix, obj, out):
    if isinstance(obj, dict):
        for key, value in obj.items():
            flatten_json(f'{prefix}.{key}' if prefix else key, value, out)
    elif isinstance(obj, list):
        out[prefix] = json.dumps(obj)
    else:
        out[prefix] = obj

def load_json(path):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text())
    return None

def collect_ablation(ablation, status):
    run_dir = Path(status['run_dir'])
    row = {
        'name': ablation['name'],
        'description': ablation['description'],
        'family': ablation.get('family'),
        'graph_factor': ablation.get('graph_factor'),
        'objective_factor': ablation.get('objective_factor'),
        'regime_factor': ablation.get('regime_factor'),
        'update_frequency_months': ablation.get('update_frequency_months'),
        'corr_lookback_days': ablation.get('corr_lookback_days'),
        'top_k': ablation.get('top_k'),
        'top_k_metric': ablation.get('top_k_metric'),
        'edge_feature_set': ablation.get('edge_feature_set'),
        'loss_type': ablation.get('loss_type'),
        'ic_loss_alpha': ablation.get('ic_loss_alpha'),
        'label_type': ablation.get('label_type'),
        'label_t': ablation.get('label_t'),
        'selection_metric': ablation.get('selection_metric'),
        'drop_edge_p': ablation.get('drop_edge_p'),
        'seed': ablation.get('seed'),
        'returncode': status['returncode'],
        'skipped': status.get('skipped', False),
        'run_dir': str(run_dir),
        'overrides': ' '.join(ablation['overrides']),
    }
    for filename, prefix in [
        ('training_summary.json', 'training'),
        ('evaluation_summary.json', 'evaluation'),
        ('run_metadata.json', 'metadata'),
        ('walkforward_summary.json', 'walkforward'),
    ]:
        data = load_json(run_dir / filename)
        if data is not None:
            flatten_json(prefix, data, row)
    return row

def add_decision_columns(df):
    out = df.copy()
    numeric_cols = [
        'evaluation.metrics.avg_ic',
        'evaluation.metrics.avg_ic_ci_lower',
        'evaluation.metrics.avg_spearman_corr',
        'evaluation.metrics.sharpe_top_20_newey_west',
        'evaluation.metrics.return_top_20',
        'evaluation.metrics.top_20_return_ci_lower',
        'evaluation.metrics.hit_rate',
        'evaluation.metrics.long_short_spread',
        'training.mean_best_val_ic',
        'training.models_trained',
        'ic_loss_alpha',
        'label_t',
        'drop_edge_p',
        'seed',
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    score = pd.Series(0.0, index=out.index)
    weights = {
        'evaluation.metrics.avg_ic': 0.35,
        'evaluation.metrics.avg_spearman_corr': 0.25,
        'evaluation.metrics.sharpe_top_20_newey_west': 0.25,
        'evaluation.metrics.return_top_20': 0.15,
    }
    for col, weight in weights.items():
        if col in out.columns:
            vals = out[col].astype(float)
            denom = vals.std(skipna=True)
            if denom and np.isfinite(denom) and denom > 0:
                score += weight * ((vals - vals.mean(skipna=True)) / denom).fillna(0.0)
    out['decision_score'] = score
    out['status'] = np.where(out['returncode'].eq(0), 'OK', 'FAILED')
    if 'evaluation.metrics.avg_ic_ci_lower' in out.columns:
        out['ic_ci_pass'] = out['evaluation.metrics.avg_ic_ci_lower'].gt(0)
    if 'evaluation.metrics.top_20_return_ci_lower' in out.columns:
        out['top20_ci_pass'] = out['evaluation.metrics.top_20_return_ci_lower'].gt(0)
    return out.sort_values(['status', 'decision_score'], ascending=[True, False])

## 6. Run Recommended Confirmation Matrix

In [ ]:
manifest_path = ABLATION_ROOT / 'recommended_confirmation_ablation_manifest.json'
manifest_path.write_text(json.dumps({
    'ablations': ABLATIONS,
    'base_overrides': BASE_OVERRIDES,
    'filters': {
        'run_only_families': RUN_ONLY_FAMILIES,
        'run_name_contains': RUN_NAME_CONTAINS,
        'skip_completed_runs': SKIP_COMPLETED_RUNS,
    },
    'factor_counts': {
        'primary_candidates': len(PRIMARY_CANDIDATES),
        'seeds': len(SEEDS),
        'edge_dropout_probes': sum(a['family'] == 'edge_dropout_probe' for a in ABLATIONS),
        'total_runs': len(ABLATIONS),
    },
    'budget': {
        'num_models': NUM_MODELS,
        'num_epochs': NUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'bootstrap_resamples': BOOTSTRAP_RESAMPLES,
    },
}, indent=2))

run_queue = ABLATIONS
if RUN_ONLY_FAMILIES:
    run_queue = [a for a in run_queue if a.get('family') in set(RUN_ONLY_FAMILIES)]
if RUN_NAME_CONTAINS:
    run_queue = [a for a in run_queue if RUN_NAME_CONTAINS in a['name']]

print('Runs after filters:', len(run_queue))

statuses = []
rows = []
for ablation in run_queue:
    status = run_ablation(ablation)
    statuses.append(status)
    rows.append(collect_ablation(ablation, status))
    interim_df = add_decision_columns(pd.DataFrame(rows))
    interim_df.to_csv(ABLATION_ROOT / 'recommended_confirmation_decision_table_interim.csv', index=False)

raw_df = pd.DataFrame(rows)
raw_path = ABLATION_ROOT / 'recommended_confirmation_results_raw.csv'
raw_df.to_csv(raw_path, index=False)

decision_df = add_decision_columns(raw_df)
decision_path = ABLATION_ROOT / 'recommended_confirmation_decision_table.csv'
decision_df.to_csv(decision_path, index=False)

html_path = ABLATION_ROOT / 'recommended_confirmation_decision_table.html'
decision_df.to_html(html_path, index=False)

print('Saved manifest:', manifest_path)
print('Saved raw results:', raw_path)
print('Saved decision table:', decision_path)
print('Saved HTML table:', html_path)

display_cols = [c for c in [
    'status',
    'name',
    'family',
    'decision_score',
    'evaluation.metrics.avg_ic',
    'evaluation.metrics.avg_ic_ci_lower',
    'evaluation.metrics.avg_spearman_corr',
    'evaluation.metrics.sharpe_top_20_newey_west',
    'evaluation.metrics.return_top_20',
    'evaluation.metrics.top_20_return_ci_lower',
    'training.mean_best_val_ic',
    'graph_factor',
    'objective_factor',
    'regime_factor',
    'selection_metric',
    'drop_edge_p',
    'seed',
    'run_dir',
] if c in decision_df.columns]
display(decision_df[display_cols])

## 7. Main Effects and Interaction Tables

In [ ]:
METRICS_FOR_EFFECTS = [
    'decision_score',
    'evaluation.metrics.avg_ic',
    'evaluation.metrics.avg_ic_ci_lower',
    'evaluation.metrics.avg_spearman_corr',
    'evaluation.metrics.return_top_20',
    'evaluation.metrics.top_20_return_ci_lower',
    'evaluation.metrics.sharpe_top_20_newey_west',
]
METRICS_FOR_EFFECTS = [c for c in METRICS_FOR_EFFECTS if c in decision_df.columns]

ok_runs = decision_df[decision_df['status'].eq('OK')].copy()
ok_primary = ok_runs[ok_runs['family'].eq('primary')].copy()

effect_tables = {}
for factor in [
    'graph_factor',
    'objective_factor',
    'loss_type',
    'regime_factor',
    'selection_metric',
    'drop_edge_p',
    'seed',
    'family',
]:
    source = ok_primary if factor in {'graph_factor', 'objective_factor', 'loss_type', 'regime_factor', 'selection_metric'} else ok_runs
    if factor in source.columns and not source.empty:
        effect = source.groupby(factor, dropna=False)[METRICS_FOR_EFFECTS].agg(['mean', 'median', 'count'])
        effect_tables[factor] = effect
        path = ABLATION_ROOT / f'{factor}_main_effects.csv'
        effect.to_csv(path)
        print('Saved:', path)
        display(effect)

if not ok_primary.empty and METRICS_FOR_EFFECTS:
    interaction_pairs = [
        ('graph_factor', 'objective_factor'),
        ('graph_factor', 'regime_factor'),
        ('objective_factor', 'regime_factor'),
    ]
    for left, right in interaction_pairs:
        if left in ok_primary.columns and right in ok_primary.columns:
            pivot = ok_primary.pivot_table(
                index=left,
                columns=right,
                values='decision_score',
                aggfunc='mean',
            )
            path = ABLATION_ROOT / f'interaction__{left}__{right}.csv'
            pivot.to_csv(path)
            print('Saved:', path)
            display(pivot)
else:
    print('No successful primary runs available for effect tables yet.')

## 8. Visualize The Decision Table

In [ ]:
import matplotlib.pyplot as plt

plot_df = decision_df[decision_df['status'].eq('OK')].copy()
metric_cols = [
    'decision_score',
    'evaluation.metrics.avg_ic',
    'evaluation.metrics.avg_spearman_corr',
    'evaluation.metrics.sharpe_top_20_newey_west',
    'evaluation.metrics.return_top_20',
]
metric_cols = [c for c in metric_cols if c in plot_df.columns]

if not plot_df.empty and metric_cols:
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(18, 4 * len(metric_cols)))
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, metric_cols):
        ordered = plot_df.sort_values(col, ascending=False)
        ax.bar(ordered['name'], ordered[col])
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=75)
        ax.axhline(0, color='black', linewidth=0.8, alpha=0.4)
    plt.tight_layout()
    plot_path = ABLATION_ROOT / 'ablation_metric_bars.png'
    plt.savefig(plot_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved plot:', plot_path)
else:
    print('No successful runs or numeric metrics available to plot.')

## 9. Inspect Failed Runs

In [ ]:
failed = [s for s in statuses if s['returncode'] != 0]
print('Failed runs:', len(failed))
for status in failed:
    run_dir = Path(status['run_dir'])
    print('\n' + '=' * 100)
    print(status['name'])
    print('Run dir:', run_dir)
    stderr_path = run_dir / 'stderr.log'
    stdout_path = run_dir / 'stdout.log'
    if stderr_path.exists():
        print('stderr tail:')
        print(stderr_path.read_text()[-4000:])
    if stdout_path.exists():
        print('stdout tail:')
        print(stdout_path.read_text()[-4000:])

## 10. Export Summary Report

In [ ]:
report_path = ABLATION_ROOT / 'recommended_confirmation_ablation_summary_report.md'
top_rows = decision_df.head(10).copy()
report_lines = [
    '# MCI-GRU Recommended Confirmation Ablation Summary',
    '',
    f'Run root: `{ABLATION_ROOT}`',
    f'Data file: `{DATA_FILE}`',
    f'Budget: {NUM_MODELS} models, {NUM_EPOCHS} epochs, early stopping patience {EARLY_STOPPING_PATIENCE}, {BOOTSTRAP_RESAMPLES} bootstrap resamples.',
    f'Seeds: `{SEEDS}`',
    '',
    '## Top Runs',
    '',
    top_rows[[c for c in [
        'status',
        'name',
        'family',
        'decision_score',
        'evaluation.metrics.avg_ic',
        'evaluation.metrics.avg_ic_ci_lower',
        'evaluation.metrics.return_top_20',
        'evaluation.metrics.top_20_return_ci_lower',
        'graph_factor',
        'objective_factor',
        'regime_factor',
        'selection_metric',
        'drop_edge_p',
        'seed',
        'run_dir',
    ] if c in top_rows.columns]].to_markdown(index=False),
    '',
    '## Interpretation Notes',
    '',
    '- All primary candidates use raw 5-day return labels, so portfolio-return metrics are on a comparable scale.',
    '- Static threshold is the robust baseline; dynamic 6-month/504-day threshold is the graph challenger.',
    '- Pure IC is the ranking-objective challenger; combined alpha 0.75 with validation-loss selection is the stability control.',
    '- Re-run with multiple seeds before promoting a production default.',
    '',
    '## Output Files',
    '',
    f'- Manifest: `{manifest_path}`',
    f'- Raw results: `{raw_path}`',
    f'- Decision table: `{decision_path}`',
    f'- HTML table: `{html_path}`',
    f'- Metric bars: `{ABLATION_ROOT / "ablation_metric_bars.png"}`',
]
report_path.write_text('\n'.join(report_lines))
print('Saved report:', report_path)
print(report_path.read_text()[:4000])